---
title: Introduction to primitives
description: Introduction to primitives in Qiskit Runtime, and an explanation of available primitives
---


# Introduction to primitives

{/*
  DO NOT EDIT THIS CELL!!!
  This cell's content is generated automatically by a script. Anything you add
  here will be removed next time the notebook is run. To add new content, create
  a new cell before or after this one.
*/}

<Accordion>
<AccordionItem title="Package versions">

The code on this page was developed using the following requirements.
We recommend using these versions or newer.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</AccordionItem>
</Accordion>

<Admonition type="note" title="New execution model, now in beta release">
The beta release of a new execution model is now available. The directed execution model provides more flexibility when customizing your error mitigation workflow. See the [Directed execution model](/docs/guides/directed-execution-model) guide for more information.
</Admonition>

Primitives were created to simplify the most common tasks for quantum computers. Namely, sampling quantum states and calculating expectation values. The Qiskit Runtime primitives ([`EstimatorV2`](/docs/api/qiskit-ibm-runtime/estimator-v2) and [`SamplerV2`](/docs/api/qiskit-ibm-runtime/sampler-v2)) are implementations of the Qiskit [primitives base classes](/docs/guides/primitives). They provide a more sophisticated implementation (for example, by including error mitigation) as a cloud-based service and are used to access IBM Quantum&reg; hardware.

<span id="estimator"></span>
### Estimator

The Estimator primitive computes the expectation values for one or more observables with respect to states prepared by quantum circuits. The circuits can be parametrized, as long as the parameter values are also provided as input to the primitive.

The input is an array of [PUBs](/docs/guides/primitive-input-output#pubs). Each PUB is in the format:

(`<single circuit>`, `<one or more observables>`, `<optional one or more parameter values>`, `<optional precision>`),

where the optional `parameter values` can be a list or a single parameter.  If the input contains measurements, they are ignored.

### Estimator

The Estimator primitive computes the expectation values for one or more observables with respect to states prepared by quantum circuits. The circuits can be parametrized, as long as the parameter values are also provided as input to the primitive.

The input is an array of [PUBs](/docs/guides/primitive-input-output#pubs). Each PUB is in the format:

(`<single circuit>`, `<one or more observables>`, `<optional one or more parameter values>`, `<optional precision>`),

where the optional `parameter values` can be a list or a single parameter.  If the input contains measurements, they are ignored.

The output is a [`PubResult`](/docs/api/qiskit/qiskit.primitives.PubResult#pubresult), with both data and metadata.  The data contains at least an array of expectation values (`PubResult.data.evs`) and associated standard deviations (either `PubResult.data.stds` or `PubResult.data.ensemble_standard_error` depending on the `resilience_level` used), but can contain more data depending on the error mitigation options that were specified.

The Estimator combines elements from observables and parameter values by following NumPy broadcasting rules as described in the [Primitive inputs and outputs](primitive-input-output#broadcasting-rules) topic.

#### How the Estimator calculates error

In addition to the estimate of the mean of the observables passed in the input PUBs (the `evs` field of the `DataBin`), the Estimator also attempts to deliver an estimate of the error associated with those expectation values. All estimator queries will populate the `stds` field with a quantity like the standard error of the mean for each expectation value, but some error mitigation options produce additional information, such as `ensemble_standard_error`.

Consider a single observable $\mathcal{O}$. In the absence of [ZNE](/docs/guides/error-mitigation-and-suppression-techniques#zero-noise-extrapolation-zne), you can think of each shot of the Estimator execution as providing a point estimate of the expectation value $\langle \mathcal{O} \rangle$. If the pointwise estimates are in a vector `Os`, then the value returned in `ensemble_standard_error` is equivalent to the following (in which $\sigma_{\mathcal{O}}$ is the [standard deviation of the expectation value](/docs/api/qiskit/qiskit.primitives.BackendEstimatorV2) estimate and $N_{shots}$ is the number of shots):

$$\frac{ \sigma_{\mathcal{O}} }{ \sqrt{N_{shots}} },$$

which treats all shots as part of a single ensemble. If you requested gate [twirling](/docs/guides/error-mitigation-and-suppression-techniques#pauli-twirling) (`twirling.enable_gates = True`), you can sort the pointwise estimates of $\langle \mathcal{O} \rangle$ into sets that share a common twirl. Call these sets of estimates `O_twirls`, and there are `num_randomizations` (number of twirls) of them. Then `stds` is the standard error of the mean of `O_twirls`, as in

$$\frac{ \sigma_{\mathcal{O}} }{ \sqrt{N_{twirls}} },$$

where $\sigma_{\mathcal{O}}$ is the standard deviation of `O_twirls` and $N_{twirls}$ is the number of twirls. When you do not enable twirling, `stds` and `ensemble_standard_error` are equal.

If you enable ZNE, then the `stds` described above become weights in a non-linear regression to an extrapolator model. What finally gets returned in the `stds` in this case is the uncertainty of the fit model evaluated at a noise factor of zero. When there is a poor fit, or large uncertainty in the fit, the reported `stds` can become very large. When ZNE is enabled, `pub_result.data.evs_noise_factors` and `pub_result.data.stds_noise_factors` are also populated, so that you can do your own extrapolation.

<span id="sampler"></span>
### Sampler

The Sampler's core task is sampling the output register from the execution of one or more quantum circuits. The input circuits can be parametrized, as long as the parameter values are also provided as input to the primitive.

The input is one or more [PUBs,](/docs/guides/primitive-input-output#pubs) in the format:

(`<single circuit>`, `<one or more optional parameter value>`, `<optional shots>`),

where there can be multiple `parameter values` items, and each item can be either an array or a single parameter, depending on the chosen circuit. Additionally, the input must contain measurements.

The output is counts or per-shot measurements, as [`PubResult`](/docs/api/qiskit/qiskit.primitives.PubResult#pubresult) objects, without weights. The result class, however, has methods to return weighted samples, such as counts. See [Primitive inputs and outputs](primitive-input-output#broadcasting-rules) for full details.  The Sampler result metadata also includes execution timing information called the [_execution span_](monitor-job#execution-spans).

## Next steps

<Admonition type="tip" title="Recommendations">
    - Learn about the [Qiskit primitives](/docs/guides/primitives) that the Qiskit Runtime primitives are based on.
    - Read [Get started with primitives](/docs/guides/get-started-with-primitives) to implement primitives in your work.
    - Review detailed [primitives examples](/docs/guides/primitives-examples).
    - See [Primitive inputs and outputs](primitive-input-output#broadcasting-rules) for detailed information.
    - Practice with primitives by working through the [Cost function lesson](/learning/courses/variational-algorithm-design/cost-functions) in IBM Quantum Learning.
    - See the [EstimatorV2 API reference](/docs/api/qiskit-ibm-runtime/estimator-v2) and [SamplerV2 API reference](/docs/api/qiskit-ibm-runtime/sampler-v2).
</Admonition>